# Улучшение качества видео — бесплатный GPU

Самостоятельный notebook для Kaggle / Google Colab. Никаких API-серверов и платных
ресурсов. CUDA обязательна. Это экспериментальный framewise RealESRGAN, а не
гарантированное восстановление: проверь мерцание, лица и уже вшитые буквы.
GPU-исполнение этого пакета ещё не проверено в реальной GPU-сессии.

1. Выбери бесплатный GPU; на Kaggle включи Internet для установки библиотек/весов.
2. Выполняй ячейки по порядку, загрузи оригинал и подтверди права.
3. Сначала sample 3 секунды, затем просмотри отдельные before/after и comparison.
4. Только после принятия пробы запускай последнюю ячейку для полного видео.

Исходник не перезаписывается. На Kaggle используй **private** notebook и private
input dataset. Не публикуй notebook с личным видео. Для входа/скачивания с телефона
может понадобиться режим браузера «Версия для ПК». Бесплатный GPU не гарантирован.


In [ ]:
import subprocess, sys, shutil
import torch
assert torch.cuda.is_available(), 'Бесплатный GPU не выделен. Стоп, платный billing не включать.'
print('GPU:', torch.cuda.get_device_name(0))
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), 'Нужны FFmpeg и FFprobe в среде.'
subprocess.run([sys.executable,'-m','pip','install','spandrel==0.4.1','Pillow==11.3.0'],check=True)
print('torch:', torch.__version__)


In [ ]:
from pathlib import Path
WORK = Path("/kaggle/working/video-quality" if Path("/kaggle/working").exists() else "/content/video-quality")
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'run.py').write_text('#!/usr/bin/env python3\n"""Video quality pipeline. Standard library + native FFmpeg; no API calls."""\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nfrom fractions import Fraction\n\nPROFILES = {\n    \'baseline\': \'\',\n    \'gentle\': \'unsharp=5:5:0.30:5:5:0\',\n    \'denoise\': \'hqdn3d=1.0:0.75:1.5:1.0,unsharp=5:5:0.25:5:5:0\',\n    \'lanczos\': \'\',\n}\n\ndef execute(args, timeout=900):\n    r = subprocess.run([str(x) for x in args], capture_output=True, text=True,\n                       timeout=timeout)\n    if r.returncode:\n        # Do not log untrusted file metadata / remote URL in a CI error.\n        raise ValueError(f\'{Path(str(args[0])).name} failed, exit={r.returncode}\')\n    return r.stdout\n\ndef ff(*args):\n    return execute([\'ffmpeg\', \'-hide_banner\', \'-v\', \'error\', \'-nostdin\',\n                    \'-threads\', \'1\', \'-filter_threads\', \'1\', \'-filter_complex_threads\', \'1\', *args])\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(b)\n    return h.hexdigest()\n\ndef probe(path):\n    return json.loads(execute([\'ffprobe\', \'-v\', \'error\', \'-show_streams\',\n                              \'-show_format\', \'-of\', \'json\', path]))\n\ndef video(info):\n    streams = [s for s in info[\'streams\'] if s[\'codec_type\'] == \'video\'\n               and not s.get(\'disposition\', {}).get(\'attached_pic\')]\n    if len(streams) != 1:\n        raise ValueError(\'Exactly one video stream required\')\n    return streams[0]\n\ndef duration(info):\n    n = float(info[\'format\'][\'duration\'])\n    if not math.isfinite(n) or not 0 < n <= 180:\n        raise ValueError(\'Supported duration: >0 and <=180 seconds\')\n    return n\n\ndef rate(stream):\n    return float(Fraction(stream.get(\'avg_frame_rate\', \'0/1\')))\n\ndef validate_media(info, processing=False):\n    s = video(info)\n    duration(info)\n    if s[\'width\'] * s[\'height\'] > 4096 * 4096:\n        raise ValueError(\'Input exceeds pixel limit\')\n    if processing:\n        if len([x for x in info[\'streams\'] if x[\'codec_type\'] == \'audio\']) > 1:\n            raise ValueError(\'Multiple audio tracks: choose a track explicitly before processing\')\n        if s.get(\'color_transfer\') in (\'smpte2084\', \'arib-std-b67\'):\n            raise ValueError(\'HDR needs a dedicated color-managed conversion; STOP\')\n        if s.get(\'sample_aspect_ratio\', \'1:1\') not in (\'1:1\', \'0:1\', \'N/A\'):\n            raise ValueError(\'Non-square pixels: explicit conversion required\')\n        rotation = float(s.get(\'tags\', {}).get(\'rotate\', 0))\n        if rotation or any(float(x.get(\'rotation\', 0)) for x in s.get(\'side_data_list\', [])):\n            raise ValueError(\'Rotated input: normalize orientation explicitly first\')\n        if s[\'width\'] % 2 or s[\'height\'] % 2:\n            raise ValueError(\'Odd dimensions: explicit framing decision required\')\n        if not 0 < rate(s) <= 120:\n            raise ValueError(\'Unsupported frame rate\')\n        if s.get(\'field_order\', \'progressive\') not in (\'progressive\', \'unknown\'):\n            raise ValueError(\'Interlaced input requires a dedicated deinterlace decision\')\n\ndef metadata(info):\n    # Whitelist technical fields; omit file path, tags, device/author metadata.\n    fields = [\'codec_type\', \'codec_name\', \'width\', \'height\', \'pix_fmt\',\n              \'r_frame_rate\', \'avg_frame_rate\', \'sample_aspect_ratio\',\n              \'color_space\', \'color_transfer\', \'color_primaries\', \'color_range\',\n              \'sample_rate\', \'channels\', \'bit_rate\', \'field_order\']\n    return {\'duration\': duration(info), \'size\': int(info[\'format\'][\'size\']),\n            \'streams\': [{k: s[k] for k in fields if k in s} for s in info[\'streams\']]}\n\ndef settings(profile, width):\n    if profile not in PROFILES:\n        raise ValueError(\'Unknown profile\')\n    if width and (not 16 <= width <= 2160 or width % 2):\n        raise ValueError(\'Target width must be even, 16..2160; 0 preserves dimensions\')\n    return {\'version\': 1, \'profile\': profile, \'width\': width,\n            \'filter\': PROFILES[profile], \'crf\': 18, \'codec\': \'libx264\'}\n\ndef output_dir(path, source=None):\n    p = Path(path).resolve()\n    if source and (p == Path(source).resolve() or p in Path(source).resolve().parents):\n        raise ValueError(\'Output must not contain the input\')\n    if p.exists() and any(p.iterdir()):\n        raise ValueError(\'Output directory must be new or empty\')\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\ndef write_json(path, data):\n    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2) + \'\\n\')\n\ndef filters(config, baseline=False):\n    parts = []\n    if not baseline and config[\'filter\']:\n        parts.append(config[\'filter\'])\n    if config[\'width\']:\n        parts.append(f"scale={config[\'width\']}:-2:flags=lanczos")\n    parts.extend([\'setsar=1\', \'format=yuv420p\'])\n    return \',\'.join(parts)\n\ndef encode(source, out, config, start=0, length=None, baseline=False):\n    args = [\'-n\', \'-i\', source, \'-ss\', str(start)]\n    if length is not None:\n        args += [\'-t\', str(length)]\n    ff(*args, \'-map\', \'0:v:0\', \'-map\', \'0:a:0?\', \'-vf\', filters(config, baseline),\n       \'-fps_mode\', \'passthrough\', \'-c:v\', \'libx264\', \'-threads\', \'2\',\n       \'-preset\', \'medium\', \'-crf\', \'18\', \'-c:a\', \'aac\', \'-b:a\', \'192k\',\n       \'-map_metadata\', \'-1\', \'-movflags\', \'+faststart\', out)\n\ndef verify(out, source_info, expected_duration, width=0):\n    info = probe(out)\n    v, original = video(info), video(source_info)\n    if v[\'codec_name\'] != \'h264\' or v[\'pix_fmt\'] != \'yuv420p\':\n        raise ValueError(\'Output codec/pixel format mismatch\')\n    target_w = width or original[\'width\']\n    if v[\'width\'] != target_w:\n        raise ValueError(\'Output width mismatch\')\n    if abs(v[\'height\'] / v[\'width\'] - original[\'height\'] / original[\'width\']) > 2 / target_w:\n        raise ValueError(\'Output aspect ratio changed\')\n    if v.get(\'sample_aspect_ratio\', \'1:1\') != \'1:1\':\n        raise ValueError(\'Output SAR mismatch\')\n    if abs(duration(info) - expected_duration) > max(.12, 2 / rate(original)):\n        raise ValueError(\'Output duration mismatch\')\n    if abs(rate(v) - rate(original)) > max(.05, rate(original) * .01):\n        raise ValueError(\'Output frame rate drift\')\n    input_audio = [s for s in source_info[\'streams\'] if s[\'codec_type\'] == \'audio\']\n    output_audio = [s for s in info[\'streams\'] if s[\'codec_type\'] == \'audio\']\n    if bool(input_audio) != bool(output_audio):\n        raise ValueError(\'Audio presence mismatch\')\n    if output_audio and output_audio[0][\'codec_name\'] != \'aac\':\n        raise ValueError(\'Output audio must be AAC\')\n    ff(\'-xerror\', \'-i\', out, \'-map\', \'0:v:0\', \'-map\', \'0:a:0?\', \'-f\', \'null\', \'-\')\n    return metadata(info)\n\ndef comparison(before, after, out):\n    # Identical time range, both shown at 540px width. Sound from before only.\n    ff(\'-n\', \'-i\', before, \'-i\', after, \'-filter_complex\',\n       \'[0:v]scale=540:-2,setsar=1[a];[1:v]scale=540:-2,setsar=1[b];[a][b]hstack=inputs=2[v]\',\n       \'-map\', \'[v]\', \'-map\', \'0:a:0?\', \'-c:v\', \'libx264\', \'-threads\', \'2\',\n       \'-crf\', \'18\', \'-c:a\', \'aac\', \'-map_metadata\', \'-1\', \'-movflags\', \'+faststart\', out)\n\ndef check_approval(manifest, source_sha, config, accept):\n    if not accept:\n        raise ValueError(\'Final requires explicit acceptance of the sample\')\n    if manifest.get(\'stage\') != \'sample\' or manifest.get(\'source_sha256\') != source_sha:\n        raise ValueError(\'Sample belongs to a different source or stage\')\n    if manifest.get(\'settings\') != config or manifest.get(\'technical_qa\') != \'PASS\':\n        raise ValueError(\'Sample settings/technical QA mismatch\')\n    if not manifest.get(\'after_sha256\'):\n        raise ValueError(\'Missing sample fingerprint\')\n\ndef process(args):\n    if not re.fullmatch(r\'VQ-\\d{6}-\\d{2,3}\', args.job_id):\n        raise ValueError(\'Use technical job ID VQ-YYYYMM-NN\')\n    source = Path(args.input).resolve()\n    if not source.is_file() or source.stat().st_size > 500 * 1024 * 1024:\n        raise ValueError(\'Input missing or exceeds 500 MiB\')\n    info = probe(source)\n    validate_media(info, args.mode != \'diagnose\')\n    source_sha = digest(source)\n    cfg = settings(args.profile, args.width)\n    if args.mode != \'diagnose\' and not args.rights_confirmed:\n        raise ValueError(\'Rights/consents must be confirmed before processing\')\n    if args.mode == \'final\':\n        if not args.approval:\n            raise ValueError(\'Final requires a sample manifest\')\n        accepted = json.loads(Path(args.approval).read_text())\n        if accepted.get(\'job_id\') != args.job_id:\n            raise ValueError(\'Sample belongs to another quality job\')\n        check_approval(accepted, source_sha, cfg, args.accept_sample)\n    if args.mode == \'sample\' and (not math.isfinite(args.start) or\n            not math.isfinite(args.seconds) or args.start < 0 or\n            not 3 <= args.seconds <= 5 or args.start + args.seconds > duration(info) + .001):\n        raise ValueError(\'Sample must be a valid 3–5 second range inside source\')\n    out = output_dir(args.output, source)\n    manifest = {\'job_id\': args.job_id, \'stage\': args.mode, \'source_sha256\': source_sha,\n                \'source\': metadata(info), \'settings\': cfg,\n                \'ffmpeg_version\': execute([\'ffmpeg\', \'-version\']).splitlines()[0],\n                \'status\': \'NEEDS_VISUAL_REVIEW\', \'visual_review\': \'NOT_PERFORMED\',\n                \'audio_listening\': \'NOT_PERFORMED\'}\n    ff(\'-xerror\', \'-i\', source, \'-map\', \'0:v:0\', \'-map\', \'0:a:0?\', \'-f\', \'null\', \'-\')\n    if args.mode == \'diagnose\':\n        for i in range(5):\n            t = max(0, duration(info) * (i + .5) / 5)\n            ff(\'-n\', \'-ss\', str(t), \'-i\', source, \'-frames:v\', \'1\',\n               \'-vf\', \'scale=540:-2\', \'-update\', \'1\', out / f\'frame-{i+1}.jpg\')\n        manifest[\'technical_qa\'] = \'PASS\'\n        manifest[\'observations\'] = [\n            \'Metadata and frames do not establish motion quality or source resolution.\',\n            \'Inspect faces, motion blur, compression blocks, lighting, text and upload copy.\',\n            \'Compare original camera file, this export and the social-platform download.\',\n            \'Do not infer a quality score from bitrate or pixel dimensions.\']\n    elif args.mode == \'sample\':\n        encode(source, out / \'before.mp4\', cfg, args.start, args.seconds, baseline=True)\n        encode(source, out / \'after.mp4\', cfg, args.start, args.seconds)\n        verify(out / \'before.mp4\', info, args.seconds, args.width)\n        manifest[\'output\'] = verify(out / \'after.mp4\', info, args.seconds, args.width)\n        comparison(out / \'before.mp4\', out / \'after.mp4\', out / \'comparison.mp4\')\n        ff(\'-xerror\', \'-i\', out / \'comparison.mp4\', \'-f\', \'null\', \'-\')\n        manifest.update({\'technical_qa\': \'PASS\', \'start\': args.start,\n                         \'seconds\': args.seconds, \'after_sha256\': digest(out / \'after.mp4\'),\n                         \'comparison_layout\': \'LEFT: baseline, RIGHT: candidate; preview only\'})\n    else:\n        encode(source, out / \'final.mp4\', cfg)\n        manifest[\'output\'] = verify(out / \'final.mp4\', info, duration(info), args.width)\n        manifest.update({\'technical_qa\': \'PASS\', \'output_sha256\': digest(out / \'final.mp4\'),\n                         \'sample_accepted_by\': os.environ.get(\'GITHUB_ACTOR\', \'local operator\'),\n                         \'accepted_sample_sha256\': json.loads(Path(args.approval).read_text())[\'after_sha256\']})\n    write_json(out / \'manifest.json\', manifest)\n    (out / \'REVIEW.md\').write_text(\n        \'# Проверка качества\\n\\nСтатус: NEEDS_VISUAL_REVIEW.\\n\'\n        \'Техническая проверка не доказывает улучшение.\\n\'\n        \'В comparison.mp4 слева baseline, справа candidate. Смотри также оба файла \'\n        \'в полном размере.\\n\\nПроверь: лица/кожу, сетку/мелкие детали, движение, \'\n        \'мерцание, ореолы, буквы, цвет, звук и синхронизацию.\\n\'\n        \'Решение: принять / отклонить / оставить оригинал. Причина: ____.\\n\')\n    return manifest\n\ndef parser():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--mode\', choices=[\'diagnose\', \'sample\', \'final\'], required=True)\n    p.add_argument(\'--input\', required=True)\n    p.add_argument(\'--output\', required=True)\n    p.add_argument(\'--job-id\', required=True)\n    p.add_argument(\'--profile\', choices=PROFILES, default=\'gentle\')\n    p.add_argument(\'--width\', type=int, default=0)\n    p.add_argument(\'--start\', type=float, default=0)\n    p.add_argument(\'--seconds\', type=float, default=3)\n    p.add_argument(\'--rights-confirmed\', action=\'store_true\')\n    p.add_argument(\'--approval\')\n    p.add_argument(\'--accept-sample\', action=\'store_true\')\n    return p\n\nif __name__ == \'__main__\':\n    try:\n        m = process(parser().parse_args())\n        print(json.dumps({\'stage\': m[\'stage\'], \'technical_qa\': m[\'technical_qa\'],\n                          \'status\': m[\'status\']}))\n    except (ValueError, KeyError, OSError, subprocess.SubprocessError) as e:\n        print(\'BLOCKED: \' + str(e), file=sys.stderr)\n        sys.exit(1)\n')
(WORK / 'gpu.py').write_text('"""Optional interactive Kaggle/Colab RealESRGAN trial. No paid APIs or face repair."""\nimport argparse\nfrom fractions import Fraction\nimport json\nfrom pathlib import Path\nimport shutil\nimport re\nimport math\nimport subprocess\nimport urllib.request\nimport run as q\n\nMODEL_URL = \'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth\'\n\ndef cfr_check(source, fps):\n    info = json.loads(q.execute([\'ffprobe\', \'-v\', \'error\', \'-select_streams\', \'v:0\',\n        \'-show_entries\', \'frame=best_effort_timestamp_time\', \'-of\', \'json\', source]))\n    ts = [float(x[\'best_effort_timestamp_time\']) for x in info[\'frames\']\n          if \'best_effort_timestamp_time\' in x]\n    if len(ts) < 2 or any(abs((b-a) - 1/fps) > .001 for a,b in zip(ts, ts[1:])):\n        raise ValueError(\'GPU path requires CFR. VFR needs an explicit timing conversion first.\')\n\ndef main(a):\n    import numpy as np\n    from PIL import Image\n    import torch\n    from spandrel import ModelLoader, ImageModelDescriptor\n    if not torch.cuda.is_available():\n        raise ValueError(\'CUDA GPU unavailable. Stop; do not activate paid resources.\')\n    if not a.rights_confirmed:\n        raise ValueError(\'Confirm rights and permission to upload to this platform first\')\n    if not re.fullmatch(r\'VQ-\\d{6}-\\d{2,3}\', a.job_id):\n        raise ValueError(\'Use VQ-YYYYMM-NN quality job ID\')\n    if not 0 <= a.strength <= 1:\n        raise ValueError(\'Strength must be 0..1\')\n    source = Path(a.input).resolve()\n    if not source.is_file() or source.stat().st_size > 500 * 1024 * 1024:\n        raise ValueError(\'Input missing or exceeds size limit\')\n    info = q.probe(source)\n    q.validate_media(info, processing=True)\n    v = q.video(info)\n    fps = q.rate(v)\n    cfr_check(source, fps)\n    cfg = {\'version\': 1, \'profile\': \'realesrgan-x2plus\', \'width\': a.width,\n           \'strength\': a.strength, \'tile\': 192, \'padding\': 24,\n           \'model_url\': MODEL_URL, \'spandrel\': \'0.4.1\'}\n    q.settings(\'baseline\', a.width)  # validate width contract\n    sha = q.digest(source)\n    if a.mode == \'final\':\n        if not a.approval: raise ValueError(\'Sample manifest required\')\n        accepted = json.loads(Path(a.approval).read_text())\n        if accepted.get(\'job_id\') != a.job_id:\n            raise ValueError(\'Sample belongs to another quality job\')\n        q.check_approval(accepted, sha, cfg, a.accept_sample)\n    elif (not math.isfinite(a.start) or not math.isfinite(a.seconds) or\n          not 3 <= a.seconds <= 5 or a.start < 0 or\n          a.start + a.seconds > q.duration(info)):\n        raise ValueError(\'Choose a valid 3–5 second sample\')\n    out = q.output_dir(a.output, source)\n    weights = Path(a.weights).resolve()\n    if not weights.exists():\n        weights.parent.mkdir(parents=True, exist_ok=True)\n        urllib.request.urlretrieve(MODEL_URL, weights)\n    weight_sha = q.digest(weights)\n    if a.mode == \'final\':\n        approval = json.loads(Path(a.approval).read_text())\n        if approval.get(\'model_sha256\') != weight_sha:\n            raise ValueError(\'Model weights changed since sample; repeat sample\')\n    model = ModelLoader(device=\'cuda\').load_from_file(str(weights))\n    if not isinstance(model, ImageModelDescriptor) or model.scale != 2:\n        raise ValueError(\'Expected an x2 image restoration model\')\n    model.eval()\n    frames, processed = out/\'frames\', out/\'processed\'\n    frames.mkdir(); processed.mkdir()\n    start = a.start if a.mode == \'sample\' else 0\n    length = a.seconds if a.mode == \'sample\' else q.duration(info)\n    # Accurate trim after input; no fps interpolation. CFR checked above.\n    q.ff(\'-n\', \'-i\', source, \'-ss\', str(start), \'-t\', str(length),\n         \'-map\', \'0:v:0\', \'-fps_mode\', \'passthrough\', frames/\'%06d.png\')\n    baseline_cfg = q.settings(\'baseline\', a.width)\n    q.encode(source, out/\'before.mp4\', baseline_cfg, start, length, baseline=True)\n    for i, path in enumerate(sorted(frames.glob(\'*.png\'))):\n        image = Image.open(path).convert(\'RGB\')\n        pixels = np.asarray(image, dtype=np.float32)/255\n        h,w = pixels.shape[:2]\n        result = np.empty((h*2,w*2,3), dtype=np.uint8)\n        # Overlapping context cropped away; full tensors stay off GPU.\n        with torch.inference_mode():\n            for y in range(0,h,192):\n                for x in range(0,w,192):\n                    y0,x0 = max(0,y-24),max(0,x-24)\n                    y1,x1 = min(h,y+192+24),min(w,x+192+24)\n                    tile = torch.from_numpy(pixels[y0:y1,x0:x1].copy()).permute(2,0,1).unsqueeze(0).cuda()\n                    pred = model(tile).clamp(0,1)[0].permute(1,2,0).cpu().numpy()\n                    th,tw = min(192,h-y),min(192,w-x)\n                    crop = pred[(y-y0)*2:(y-y0+th)*2,(x-x0)*2:(x-x0+tw)*2]\n                    result[y*2:(y+th)*2,x*2:(x+tw)*2] = (crop*255+.5).astype(np.uint8)\n        tw = a.width or w\n        th = round((h*tw/w)/2)*2\n        candidate = Image.fromarray(result).resize((tw,th), Image.Resampling.LANCZOS)\n        original = image.resize((tw,th), Image.Resampling.LANCZOS)\n        Image.blend(original,candidate,a.strength).save(processed/path.name)\n        if i % 30 == 0: print(\'Processed frames:\', i+1, flush=True)\n    if not list(processed.glob(\'*.png\')): raise ValueError(\'No frames extracted\')\n    after = out/(\'after.mp4\' if a.mode == \'sample\' else \'final.mp4\')\n    q.ff(\'-n\', \'-framerate\', str(Fraction(v[\'avg_frame_rate\'])), \'-i\', processed/\'%06d.png\',\n         \'-i\', out/\'before.mp4\', \'-map\', \'0:v:0\', \'-map\', \'1:a:0?\',\n         \'-c:v\', \'libx264\', \'-threads\', \'2\', \'-crf\', \'18\', \'-pix_fmt\', \'yuv420p\',\n         \'-c:a\', \'copy\', \'-map_metadata\', \'-1\', \'-movflags\', \'+faststart\', after)\n    technical = q.verify(after, info, length, a.width)\n    if a.mode == \'sample\': q.comparison(out/\'before.mp4\', after, out/\'comparison.mp4\')\n    manifest = {\'job_id\':a.job_id, \'stage\':a.mode, \'source_sha256\':sha,\n        \'settings\':cfg, \'model_sha256\':weight_sha, \'gpu\':torch.cuda.get_device_name(0),\n        \'torch_version\':torch.__version__, \'source\':q.metadata(info), \'output\':technical,\n        \'after_sha256\':q.digest(after), \'technical_qa\':\'PASS\',\n        \'status\':\'NEEDS_VISUAL_REVIEW\', \'visual_review\':\'NOT_PERFORMED\',\n        \'audio_listening\':\'NOT_PERFORMED\', \'start\':start, \'seconds\':length,\n        \'warning\':\'Framewise model: inspect temporal flicker, faces, text, tile seams. No guaranteed recovery.\'}\n    q.write_json(out/\'manifest.json\',manifest)\n    # Only temporary frame sequences owned by this run are removed.\n    shutil.rmtree(frames); shutil.rmtree(processed)\n    print(\'Technical QA PASS. Visual review required:\', out)\n\nif __name__ == \'__main__\':\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--mode\',choices=[\'sample\',\'final\'],default=\'sample\')\n    p.add_argument(\'--input\',required=True); p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--job-id\',required=True); p.add_argument(\'--width\',type=int,default=0)\n    p.add_argument(\'--start\',type=float,default=0); p.add_argument(\'--seconds\',type=float,default=3)\n    p.add_argument(\'--strength\',type=float,default=.5)\n    p.add_argument(\'--weights\',default=\'/tmp/video-quality/RealESRGAN_x2plus.pth\')\n    p.add_argument(\'--rights-confirmed\',action=\'store_true\')\n    p.add_argument(\'--approval\'); p.add_argument(\'--accept-sample\',action=\'store_true\')\n    main(p.parse_args())\n')


In [ ]:
# Colab: появится выбор файла. Kaggle: загрузи private input через Add Input и укажи путь.
if Path('/content').exists():
    from google.colab import files
    uploaded = files.upload()
    assert len(uploaded)==1, 'Загрузи ровно один видеофайл'
    INPUT = str(Path(next(iter(uploaded))).resolve())
else:
    INPUT = '/kaggle/input/CHANGE_ME/reel_FINAL.mp4'
print('Вход нужно проверить:', Path(INPUT).name)


In [ ]:
# Поставь True только если имеешь права/согласия и разрешаешь обработку здесь.
RIGHTS_CONFIRMED = False
JOB_ID = 'VQ-202611-01'
START = 0
WIDTH = 0  # 0 сохраняет исходный размер; 2160 — только если нужен увеличенный файл
STRENGTH = 0.5  # мягкое смешивание с оригиналом
assert RIGHTS_CONFIRMED, 'Нужно подтвердить права'
assert Path(INPUT).is_file(), 'Проверь путь к файлу'
import uuid
RUN = WORK / ('run-' + uuid.uuid4().hex[:8])
COMMON = [sys.executable,str(WORK/'gpu.py'),'--input',INPUT,'--job-id',JOB_ID,
          '--width',str(WIDTH),'--strength',str(STRENGTH),'--rights-confirmed',
          '--weights',str(WORK/'RealESRGAN_x2plus.pth')]
subprocess.run(COMMON + ['--mode','sample','--output',str(RUN/'sample'),'--start',str(START)],check=True)


In [ ]:
from IPython.display import display, Video
for name in ['before.mp4','after.mp4','comparison.mp4']:
    print(name, '(comparison: слева исходный baseline, справа кандидат)')
    display(Video(str(RUN/'sample'/name),embed=True,width=360))


## Приёмка пробы

Сравни также видео в полном размере: кожа и лица, сетка/мелкие детали, быстрые
руки, вшитый текст, ореолы и мерцание. Если улучшения нет — оставь оригинал.
Полный файл технически проверяется, но просмотр и прослушивание человеком обязательны.


In [ ]:
SAMPLE_ACCEPTED = False  # True только после фактического просмотра
assert SAMPLE_ACCEPTED, 'Проба ещё не принята'
subprocess.run(COMMON + ['--mode','final','--output',str(RUN/'final'),
    '--approval',str(RUN/'sample'/'manifest.json'),'--accept-sample'],check=True)
display(Video(str(RUN/'final'/'final.mp4'),embed=True,width=360))
print('Просмотри весь файл со звуком. manifest:', RUN/'final'/'manifest.json')
if Path('/content').exists():
    from google.colab import files
    files.download(str(RUN/'final'/'final.mp4'))
else:
    print('Скачай файл из Output в Kaggle; не делай notebook публичным.')
